# Recurrence analysis

Recurrence analysis maps temporal dynamics to graphs, revealing
structure invisible to linear methods.  Unlike correlation or spectral
approaches, recurrence captures **nonlinear, non-stationary** patterns
-- exactly the kind of dynamics common in neural data.

This notebook covers fundamentals on classic signals, then applies
the full workflow to recover functional modules in a synthetic neural
population -- purely from dynamics, without any behavioral variables.

| Step | Notebook | What it does |
|---|---|---|
| **Overview** | [00 -- DRIADA overview](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/00_driada_overview.ipynb) | Core data structures, quick tour of INTENSE, DR, networks |
| Neuron analysis | [01 -- Neuron analysis](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/01_data_loading_and_neurons.ipynb) | Spike reconstruction, kinetics optimization, quality metrics, surrogates |
| Single-neuron selectivity | [02 -- INTENSE](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/02_selectivity_detection_intense.ipynb) | Detect which neurons encode which behavioral variables |
| Population geometry | [03 -- Dimensionality reduction](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/03_population_geometry_dr.ipynb) | Extract low-dimensional manifolds from population activity |
| Network analysis | [04 -- Network analysis](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/04_network_analysis.ipynb) | Build and analyze interaction graphs |
| Putting it together | [05 -- Advanced](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/05_advanced_capabilities.ipynb) | Combine INTENSE + DR, leave-one-out importance, RSA, RNN analysis |
| **Recurrence analysis** | **06 -- this notebook** | Recurrence graphs, RQA, graph representations, population modules |

**In this notebook:**

1. **Recurrence fundamentals** -- generate classic signals, select
   embedding parameters, build recurrence plots, compute RQA measures,
   compare three graph representations (RG, HVG, OPN), and track regime
   changes with windowed RQA.
2. **Population analysis** -- generate a synthetic modular population,
   build per-neuron recurrence graphs, measure pairwise Jaccard
   similarity, detect communities, and compare to ground truth.

In [ ]:
# TODO: revert to '!pip install -q driada' after v1.0.0 PyPI release
# TODO: revert @dev to @main after merging recurrence module to main
!pip install -q git+https://github.com/iabs-neuro/driada.git@dev
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import networkx as nx
import networkx.algorithms.community as nx_comm
from sklearn.metrics import adjusted_rand_score
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from driada.recurrence import (
    takens_embedding, estimate_tau, estimate_embedding_dim,
    RecurrenceGraph, plot_recurrence, compute_rqa,
    pairwise_jaccard_sparse,
)
from driada.information import TimeSeries
from driada.information import get_tdmi
from driada.experiment.synthetic import generate_tuned_selectivity_exp
from driada.network import Network

## 1. Recurrence fundamentals

### 1.1 Generate signals

We start with three signals representing the main dynamical classes
you will encounter in neural data:

- **Periodic** (sine) -- regular oscillations, like theta or gamma
  rhythms
- **Chaotic** (Lorenz attractor) -- deterministic but unpredictable,
  like population dynamics during complex behavior
- **Stochastic** (white noise) -- no temporal structure, the null
  hypothesis

Each signal is also wrapped as a [`TimeSeries`](https://driada.readthedocs.io/en/latest/api/information/core.html#driada.information.info_base.TimeSeries) so we can
use the DRIADA high-level API throughout.

In [ ]:
def _lorenz_series(n, dt=0.02, seed=42):
    """Generate x-component of Lorenz attractor via RK4 integration."""
    sigma, rho, beta = 10.0, 28.0, 8.0 / 3.0
    rng = np.random.default_rng(seed)

    def lorenz(state):
        x, y, z = state
        return np.array([sigma * (y - x), x * (rho - z) - y, x * y - beta * z])

    state = np.array([1.0, 1.0, 1.0]) + rng.normal(0, 0.01, 3)

    # Warm-up: discard transient
    warmup = 2000
    for _ in range(warmup):
        k1 = lorenz(state)
        k2 = lorenz(state + dt / 2 * k1)
        k3 = lorenz(state + dt / 2 * k2)
        k4 = lorenz(state + dt * k3)
        state = state + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

    # Collect samples
    x_series = np.empty(n)
    for i in range(n):
        k1 = lorenz(state)
        k2 = lorenz(state + dt / 2 * k1)
        k3 = lorenz(state + dt / 2 * k2)
        k4 = lorenz(state + dt * k3)
        state = state + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)
        x_series[i] = state[0]

    return x_series


rng = np.random.default_rng(42)
N = 800

t = np.arange(N)
sine = np.sin(2 * np.pi * t / 50) + rng.normal(0, 0.05, N)
lorenz_x = _lorenz_series(N, dt=0.02, seed=42)
noise = rng.normal(size=N)

signals = {
    "Sine (periodic)": sine,
    "Lorenz (chaotic)": lorenz_x,
    "Noise (stochastic)": noise,
}
signal_colors = {
    "Sine (periodic)": "#1f77b4",
    "Lorenz (chaotic)": "#d62728",
    "Noise (stochastic)": "#7f7f7f",
}

# Wrap each signal as a TimeSeries for the DRIADA API
ts_signals = {name: TimeSeries(sig) for name, sig in signals.items()}

print(f"Generated {N} points for each signal:")
for name in signals:
    sig = signals[name]
    print(f"  {name}: range [{sig.min():.2f}, {sig.max():.2f}]")

### 1.2 Embedding parameter selection

[Takens' theorem](https://en.wikipedia.org/wiki/Takens%27s_theorem)
lets us reconstruct the attractor from a single 1D observable using
**time-delay embedding**.  Two parameters are needed:

- **tau** (time delay) -- first minimum of time-delayed mutual
  information (TDMI).  In DRIADA:
  [`estimate_tau`](https://driada.readthedocs.io/en/latest/api/recurrence/embedding.html#driada.recurrence.embedding.estimate_tau)
- **dim** (embedding dimension) -- where the false nearest neighbours
  (FNN) fraction drops below 5%.  In DRIADA:
  [`estimate_embedding_dim`](https://driada.readthedocs.io/en/latest/api/recurrence/embedding.html#driada.recurrence.embedding.estimate_embedding_dim)

In practice, a single call to `ts.estimate_tau()` and
`ts.estimate_embedding_dim()` is all you need.  The diagnostic curves
below show what happens under the hood.

For noise, FNN never drops below 5 % because there is no attractor to
unfold.  The returned dim equals `max_dim` (a ceiling, not a
meaningful dimension).

In [ ]:
# Estimate parameters for each signal
params = {}
tdmi_curves = {}
fnn_curves = {}

for name, sig in signals.items():
    # TDMI curve (for diagnostic plot)
    tdmi = get_tdmi(sig, min_shift=1, max_shift=81, estimator="gcmi")
    tdmi_curves[name] = tdmi

    # Estimate tau and dim via the DRIADA API
    tau = estimate_tau(sig, max_shift=80)
    dim, fnn = estimate_embedding_dim(sig, tau=tau, max_dim=10,
                                       return_fractions=True)
    fnn_curves[name] = fnn
    params[name] = (tau, dim)
    print(f"{name}: tau={tau}, dim={dim}")

# 2x3 grid: top=TDMI, bottom=FNN
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
names = list(signals.keys())

for col, name in enumerate(names):
    tau, dim = params[name]
    color = signal_colors[name]

    # TDMI curve
    ax = axes[0, col]
    tdmi = tdmi_curves[name]
    shifts = np.arange(1, len(tdmi) + 1)
    ax.plot(shifts, tdmi, color=color, linewidth=1.5)
    ax.axvline(tau, color="k", linestyle="--", linewidth=1, alpha=0.7,
               label=f"tau = {tau}")
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("Lag (samples)")
    ax.legend(fontsize=9, loc="upper right")
    if col == 0:
        ax.set_ylabel("TDMI (bits)")

    # FNN curve
    ax = axes[1, col]
    fnn = fnn_curves[name]
    fnn_dims = [f[0] for f in fnn]
    fnn_fracs = [f[1] for f in fnn]
    ax.plot(fnn_dims, fnn_fracs, "o-", color=color, linewidth=1.5, markersize=5)
    ax.axhline(0.05, color="k", linestyle="--", linewidth=1, alpha=0.7,
               label="5% threshold")
    ax.axvline(dim, color="k", linestyle=":", linewidth=1, alpha=0.5,
               label=f"dim = {dim}")
    ax.set_xlabel("Embedding dimension")
    ax.set_ylim(-0.02, max(0.3, max(fnn_fracs) * 1.15) if fnn_fracs else 0.3)
    ax.legend(fontsize=9, loc="upper right")
    if col == 0:
        ax.set_ylabel("FNN fraction")

fig.suptitle("Embedding parameter selection: TDMI and FNN diagnostics",
             fontsize=13, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### 1.3 3D delay embedding

Delay-embedded Lorenz attractor recovers the butterfly shape from a
scalar observable -- Takens' theorem in action.  The
`takens_embedding` function returns a `(dim, N_embedded)` array.

In [ ]:
# Generate a longer Lorenz series for a cleaner attractor visualization
from mpl_toolkits.mplot3d.art3d import Line3DCollection

lorenz_long = _lorenz_series(2000, dt=0.02, seed=42)
tau_3d = estimate_tau(lorenz_long, max_shift=80)
emb = takens_embedding(lorenz_long, tau_3d, 3)
n_emb = emb.shape[1]

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# Connected trajectory colored by time
points = np.column_stack([emb[0], emb[1], emb[2]])
segments = np.array([[points[i], points[i+1]] for i in range(len(points)-1)])
colors = plt.cm.inferno(np.linspace(0, 1, len(segments)))
lc = Line3DCollection(segments, colors=colors, linewidths=0.5, alpha=0.7)
ax.add_collection3d(lc)

# Scatter points on top
ax.scatter(emb[0], emb[1], emb[2], c=np.arange(n_emb), cmap="inferno",
           s=2, alpha=0.4, zorder=2)

ax.set_xlim(emb[0].min(), emb[0].max())
ax.set_ylim(emb[1].min(), emb[1].max())
ax.set_zlim(emb[2].min(), emb[2].max())

sm = plt.cm.ScalarMappable(cmap="inferno", norm=plt.Normalize(0, n_emb))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.6, pad=0.1)
cbar.set_label("Time index", fontsize=10)

ax.set_xlabel(r"$x(t)$", fontsize=10)
ax.set_ylabel(r"$x(t + \tau)$", fontsize=10)
ax.set_zlabel(r"$x(t + 2\tau)$", fontsize=10)
ax.set_title(f"3D delay embedding of Lorenz attractor "
             f"(tau={tau_3d}, dim=3, n=2000)", fontsize=12)
ax.view_init(elev=40, azim=160)

fig.tight_layout()
plt.show()

### 1.4 Recurrence plots

A recurrence plot (RP) is a binary matrix where a dot at (i, j) means
states at times i and j are close in embedding space.  We build them
using [`TimeSeries`](https://driada.readthedocs.io/en/latest/api/information/core.html#driada.information.info_base.TimeSeries)`.recurrence_graph()`, which handles
embedding and Theiler window automatically:

- **Diagonal lines** = determinism (the system follows similar
  trajectories)
- **Vertical lines** = laminarity (the system gets trapped in a
  state)
- **Uniform scatter** = noise (no temporal structure)

See [`RecurrenceGraph`](https://driada.readthedocs.io/en/latest/api/recurrence/recurrence_graph.html#driada.recurrence.recurrence_graph.RecurrenceGraph) and [`plot_recurrence`](https://driada.readthedocs.io/en/latest/api/recurrence/plotting.html#driada.recurrence.plotting.plot_recurrence)
for details.

In [ ]:
# Build k-NN recurrence graphs (k=5) for all three signals
# ts.recurrence_graph() handles embedding + Theiler window automatically
graphs = {}
for name, ts in ts_signals.items():
    tau, dim = params[name]
    rg = ts.recurrence_graph(tau=tau, m=dim, k=5)
    graphs[name] = rg
    print(f"{name}: {rg.n} embedded points, {rg.adj.nnz} recurrence entries")

# 2x3 grid: top=time series, bottom=recurrence plots
names = list(signals.keys())
fig, axes = plt.subplots(2, 3, figsize=(15, 8),
                          gridspec_kw={"height_ratios": [1, 3]})

for col, name in enumerate(names):
    sig = signals[name]
    rg = graphs[name]
    color = signal_colors[name]
    n_emb = rg.n

    # Time series (top)
    ax_ts = axes[0, col]
    ax_ts.plot(np.arange(n_emb), sig[:n_emb], color=color, linewidth=0.6)
    ax_ts.set_title(name, fontsize=11)
    ax_ts.set_xlim(0, n_emb - 1)
    ax_ts.tick_params(labelbottom=False)
    if col == 0:
        ax_ts.set_ylabel("Amplitude")

    # Recurrence plot (bottom)
    ax_rp = axes[1, col]
    plot_recurrence(rg, ax=ax_rp, markersize=0.3, color="k")
    ax_rp.set_title("")
    ax_rp.set_xlim(0, n_emb - 1)
    ax_rp.set_ylim(n_emb - 1, 0)
    ax_rp.set_xlabel("Time index")
    if col == 0:
        ax_rp.set_ylabel("Time index")

fig.suptitle("Recurrence plots with marginal time series",
             fontsize=13, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

### 1.5 RQA comparison

Recurrence Quantification Analysis
([`compute_rqa`](https://driada.readthedocs.io/en/latest/api/recurrence/rqa.html#driada.recurrence.rqa.compute_rqa)) extracts diagonal and vertical line
structures from the RP.  These numbers summarize the *type* of
dynamics in a single value:

| Measure | Definition | Why it matters |
|---|---|---|
| **DET** | Fraction of recurrence in diagonal lines (l >= 2) | High DET = the system revisits similar states in predictable sequences -- a hallmark of **deterministic dynamics** |
| **LAM** | Fraction in vertical lines (v >= 2) | High LAM = the system gets **trapped** near certain states (laminar phases) |
| **ENTR** | Shannon entropy of diagonal line lengths | High ENTR = the deterministic structure is **complex** (many different recurrence lengths) |

In [ ]:
# Compute RQA for each signal
rqa_data = {}
measures = ["DET", "LAM", "ENTR"]

print(f"{'Signal':<24} {'DET':>8} {'LAM':>8} {'ENTR':>8}")
print(f"{'-' * 48}")
for name, rg in graphs.items():
    rqa = rg.rqa()
    rqa_data[name] = rqa
    print(f"{name:<24} {rqa['DET']:>8.4f} {rqa['LAM']:>8.4f} {rqa['ENTR']:>8.4f}")

# Grouped bar chart
names = list(graphs.keys())
n_measures = len(measures)
n_signals = len(names)
x = np.arange(n_measures)
width = 0.8 / n_signals

fig, ax = plt.subplots(figsize=(8, 5))
for i, name in enumerate(names):
    vals = [rqa_data[name][m] for m in measures]
    offset = (i - (n_signals - 1) / 2) * width
    ax.bar(x + offset, vals, width * 0.9,
           label=name, color=signal_colors[name], edgecolor="white",
           linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(measures, fontsize=11)
ax.set_ylabel("Value", fontsize=11)
ax.set_title("RQA measures comparison", fontsize=13)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

### 1.6 Three graph representations

DRIADA provides three ways to turn a time series into a graph.  Each
captures different aspects of the dynamics:

| Representation | Method | What it captures |
|---|---|---|
| [`RecurrenceGraph`](https://driada.readthedocs.io/en/latest/api/recurrence/recurrence_graph.html#driada.recurrence.recurrence_graph.RecurrenceGraph) | k-NN in embedding space | State-space proximity |
| [`VisibilityGraph`](https://driada.readthedocs.io/en/latest/api/recurrence/visibility.html#driada.recurrence.visibility.VisibilityGraph) | Horizontal line-of-sight | Amplitude structure |
| [`OrdinalPartitionNetwork`](https://driada.readthedocs.io/en/latest/api/recurrence/opn.html#driada.recurrence.opn.OrdinalPartitionNetwork) | Rank-pattern transitions | Temporal ordering |

All three are built via one-liner methods on [`TimeSeries`](https://driada.readthedocs.io/en/latest/api/information/core.html#driada.information.info_base.TimeSeries).

#### RecurrenceGraph

RG connects time points whose embedded states are k-nearest neighbours.
The adjacency matrix IS the recurrence plot.  Since
[`RecurrenceGraph`](https://driada.readthedocs.io/en/latest/api/recurrence/recurrence_graph.html#driada.recurrence.recurrence_graph.RecurrenceGraph) inherits from [`Network`](https://driada.readthedocs.io/en/latest/api/network/core.html#driada.network.net_base.Network),
all network analysis methods (spectral decomposition, community
detection, etc.) are available directly.

In [ ]:
from scipy.stats import rankdata

tau_sine, dim_sine = params["Sine (periodic)"]
ts_sine = ts_signals["Sine (periodic)"]

# Build RG with k=10 and a NetworkX graph for visualization
rg_sine = ts_sine.recurrence_graph(tau=tau_sine, m=dim_sine, k=10,
                                    create_nx_graph=True)

# .graph is the NetworkX graph (inherited from Network)
G_rg = rg_sine.graph

pos_rg = nx.spring_layout(G_rg, seed=42, iterations=80)
node_list = sorted(G_rg.nodes())
node_vals = rankdata(sine[node_list], method="average") / len(node_list)

fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(G_rg, pos_rg, alpha=0.1, ax=ax)
sc = nx.draw_networkx_nodes(G_rg, pos_rg, nodelist=node_list,
                             node_color=node_vals, cmap="inferno",
                             node_size=15, ax=ax)
ax.set_title("RecurrenceGraph (k=10, sine)", fontsize=12)
ax.axis("off")
plt.colorbar(sc, ax=ax, label="Signal percentile", shrink=0.8)
plt.tight_layout()
plt.show()

#### HorizontalVisibilityGraph

Two time points are connected if a horizontal line-of-sight is
unobstructed (no intermediate value exceeds both endpoints).  The
degree distribution differentiates periodic, chaotic, and random
signals (Lacasa et al. 2008).  O(N) construction via monotone stack.

In [ ]:
vg = ts_sine.visibility_graph(method="horizontal", create_nx_graph=True)

G_vg = vg.graph
pos_vg = nx.spring_layout(G_vg, seed=42, iterations=80)

node_list_vg = sorted(G_vg.nodes())
node_vals_vg = rankdata(sine[node_list_vg], method="average") / len(node_list_vg)

fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(G_vg, pos_vg, alpha=0.1, ax=ax)
sc = nx.draw_networkx_nodes(G_vg, pos_vg, nodelist=node_list_vg,
                             node_color=node_vals_vg, cmap="inferno",
                             node_size=15, ax=ax)
ax.set_title("HorizontalVisibilityGraph (sine)", fontsize=12)
ax.axis("off")
plt.colorbar(sc, ax=ax, label="Signal percentile", shrink=0.8)
plt.tight_layout()
plt.show()

#### OrdinalPartitionNetwork

Delay-embedded windows are reduced to **rank patterns** (ordinal
patterns).  Directed edges connect consecutive patterns.  Permutation
entropy measures the diversity of visited patterns -- low entropy
means the system prefers a small set of orderings.

In [ ]:
opn = ts_sine.ordinal_partition_network(tau=tau_sine, create_nx_graph=True)

G_opn = opn.graph  # directed graph (OPN transitions)
pos_opn = nx.spring_layout(G_opn, seed=42, iterations=80)

# Color OPN nodes by mean signal value of time points producing each pattern
pattern_ids = opn._pattern_ids
opn_nodes = sorted(G_opn.nodes())
opn_vals = np.zeros(len(opn_nodes))
for vi, node in enumerate(opn_nodes):
    mask = pattern_ids == node
    if mask.any():
        opn_vals[vi] = np.mean(sine[:len(pattern_ids)][mask])

fig, ax = plt.subplots(figsize=(8, 6))
nx.draw_networkx_edges(G_opn, pos_opn, alpha=0.3, ax=ax)
sc = nx.draw_networkx_nodes(G_opn, pos_opn, nodelist=opn_nodes,
                             node_color=opn_vals, cmap="inferno",
                             node_size=80, ax=ax)
ax.set_title("OrdinalPartitionNetwork (sine)", fontsize=12)
ax.axis("off")
plt.colorbar(sc, ax=ax, label="Mean signal value", shrink=0.8)

pe = ts_sine.permutation_entropy(tau=tau_sine)
print(f"Permutation entropy: {pe:.4f}")

plt.tight_layout()
plt.show()

### 1.7 Windowed RQA

Real neural recordings are **non-stationary** -- the dynamics change
as the animal moves between behavioral states.  Windowed RQA slides a
window along the RP diagonal and computes DET in each window, letting
us track regime transitions over time.

Below we concatenate sine--Lorenz--sine to simulate a regime change
and show how windowed DET drops during the chaotic segment.

In [ ]:
# Concatenate sine + Lorenz + sine (2400 points)
seg_len = N
rng2 = np.random.default_rng(99)
sine_seg1 = np.sin(2 * np.pi * np.arange(seg_len) / 50) + rng.normal(0, 0.05, seg_len)
lorenz_seg = _lorenz_series(seg_len, dt=0.02, seed=99)
sine_seg2 = np.sin(2 * np.pi * np.arange(seg_len) / 50) + rng.normal(0, 0.05, seg_len)

# Normalize Lorenz to similar amplitude range
lorenz_seg_norm = (lorenz_seg - lorenz_seg.mean()) / lorenz_seg.std()
nonstat_signal = np.concatenate([sine_seg1, lorenz_seg_norm, sine_seg2])
print(f"Non-stationary signal: {len(nonstat_signal)} points (sine-Lorenz-sine)")

# Build recurrence graph with fixed embedding parameters
nonstat_ts = TimeSeries(nonstat_signal)
nonstat_rg = nonstat_ts.recurrence_graph(tau=12, m=5, k=5)
print(f"Embedded: {nonstat_rg.n} points, {nonstat_rg.adj.nnz} recurrence entries")


def windowed_det(adj_csr, window_size, step):
    """Compute DET in sliding windows along the RP diagonal."""
    n = adj_csr.shape[0]
    positions, values = [], []
    for start in range(0, n - window_size, step):
        end = start + window_size
        sub = adj_csr[start:end, start:end]
        if sub.nnz > 0:
            rqa = compute_rqa(sub)
            values.append(rqa['DET'])
        else:
            values.append(0.0)
        positions.append((start + end) / 2)
    return np.array(positions), np.array(values)


adj_csr = nonstat_rg.adj.tocsr()
positions, det_values = windowed_det(adj_csr, window_size=150, step=10)
n_emb = nonstat_rg.n

# Plot: top=signal with colored background, bottom=windowed DET
fig, (ax_ts, ax_det) = plt.subplots(2, 1, figsize=(12, 6), sharex=True,
                                     gridspec_kw={"height_ratios": [1, 1.2]})

# Top: time series with regime shading
t_emb = np.arange(n_emb)
ax_ts.plot(t_emb, nonstat_signal[:n_emb], color="#9467bd", linewidth=0.5)

regime_colors = ["#1f77b4", "#d62728", "#1f77b4"]
regime_labels = ["Sine", "Lorenz", "Sine"]
for i in range(3):
    start = i * seg_len
    end = min((i + 1) * seg_len, n_emb)
    if start < n_emb:
        ax_ts.axvspan(start, end, alpha=0.12, color=regime_colors[i])
        mid = (start + min(end, n_emb)) / 2
        ax_ts.text(mid, ax_ts.get_ylim()[0] if i > 0 else 0, regime_labels[i],
                   ha="center", va="bottom", fontsize=9, color=regime_colors[i],
                   fontweight="bold")

ax_ts.set_ylabel("Amplitude")
ax_ts.set_title("Non-stationary signal: sine - Lorenz - sine", fontsize=12)

# Bottom: windowed DET
ax_det.plot(positions, det_values, color="#9467bd", linewidth=1.5)
ax_det.fill_between(positions, det_values, alpha=0.2, color="#9467bd")
for i in range(3):
    start = i * seg_len
    end = min((i + 1) * seg_len, n_emb)
    if start < n_emb:
        ax_det.axvspan(start, end, alpha=0.08, color=regime_colors[i])

ax_det.set_xlabel("Time index")
ax_det.set_ylabel("Windowed DET")
ax_det.set_ylim(-0.02, 1.05)
ax_det.set_title("Windowed determinism (DET) tracks regime changes", fontsize=12)
ax_det.grid(axis="y", alpha=0.3)

fig.tight_layout()
plt.show()

## 2. Recovering functional modules from population dynamics

**Scientific question:** can we identify which neurons belong to the
same functional group purely from their calcium dynamics, without
ever looking at behavioral variables?

We generate a synthetic population with **known ground-truth modules**
(120 neurons, 6 modules -- same setup as
[Notebook 04](https://colab.research.google.com/github/iabs-neuro/driada/blob/dev/notebooks/04_network_analysis.ipynb))
so we can objectively measure how well recurrence-based community
detection recovers them.

### 2.1 Generate population

We use [`generate_tuned_selectivity_exp`](https://driada.readthedocs.io/en/latest/api/experiment/synthetic.html#driada.experiment.synthetic.generators.generate_tuned_selectivity_exp) to create the
synthetic experiment directly at 5 Hz -- no manual downsampling
needed.  The `Experiment` object gives us per-neuron
[`TimeSeries`](https://driada.readthedocs.io/en/latest/api/information/core.html#driada.information.info_base.TimeSeries) via `exp.calcium.ts_list`, ready for
recurrence analysis.

In [ ]:
population = [
    {"name": "event_0", "count": 30, "features": ["event_0"]},
    {"name": "event_1", "count": 30, "features": ["event_1"]},
    {"name": "event_2", "count": 30, "features": ["event_2"]},
    {"name": "event_0|1", "count": 10, "features": ["event_0", "event_1"],
     "combination": "or"},
    {"name": "event_0|2", "count": 10, "features": ["event_0", "event_2"],
     "combination": "or"},
    {"name": "event_1|2", "count": 10, "features": ["event_1", "event_2"],
     "combination": "or"},
]

print("Generating synthetic experiment...")
exp = generate_tuned_selectivity_exp(
    population=population,
    n_discrete_features=3,
    duration=600,
    fps=5.0,
    baseline_rate=0.05,
    peak_rate=2.0,
    decay_time=2.0,
    calcium_noise=0.02,
    seed=42,
    verbose=True,
)

# exp.calcium is a MultiTimeSeries; .ts_list gives per-neuron TimeSeries
n_neurons, n_frames = exp.calcium.data.shape
ts_list = exp.calcium.ts_list

print(f"\n{n_neurons} neurons, {n_frames} frames ({exp.fps:.0f} Hz)")
for g in population:
    feat_str = " OR ".join(g["features"])
    print(f"  {g['count']:>2} {g['name']:<12} -> {feat_str}")

### 2.2 Per-neuron embedding and RQA

Each neuron's calcium trace has its own optimal embedding parameters.
We use the [`TimeSeries`](https://driada.readthedocs.io/en/latest/api/information/core.html#driada.information.info_base.TimeSeries) methods to estimate tau and dim,
then build a k-NN recurrence graph (k=50) and compute RQA.  The
recurrence graphs are cached on each `TimeSeries`, so we can reuse
them later without rebuilding.

In [ ]:
def get_neuron_modules(population):
    """Map neuron index to module name."""
    modules = {}
    idx = 0
    for group in population:
        for _ in range(group["count"]):
            modules[idx] = group["name"]
            idx += 1
    return modules


neuron_modules = get_neuron_modules(population)
groups = {}
for idx, m in neuron_modules.items():
    groups.setdefault(m, []).append(idx)
module_names = sorted(groups.keys())

print("Computing per-neuron embedding and RQA...")
taus = np.empty(n_neurons, dtype=int)
dims = np.empty(n_neurons, dtype=int)
rqa_results = []
per_neuron_rgs = []

for i, ts in enumerate(ts_list):
    taus[i] = ts.estimate_tau(max_shift=60)
    dims[i] = ts.estimate_embedding_dim(tau=taus[i], max_dim=15)
    rqa_results.append(ts.rqa(tau=taus[i], m=dims[i], k=50))
    # The RG is now cached on ts; retrieve it for later use
    per_neuron_rgs.append(ts.recurrence_graph(tau=taus[i], m=dims[i], k=50))

print(f"\n{'Module':<15} {'n':>3} {'tau':>6} {'dim':>5}"
      f" {'DET':>7} {'LAM':>7} {'ENTR':>7}")
print(f"{'-' * 55}")

for m in module_names:
    idxs = groups[m]
    det = np.array([rqa_results[i]['DET'] for i in idxs])
    lam = np.array([rqa_results[i]['LAM'] for i in idxs])
    entr = np.array([rqa_results[i]['ENTR'] for i in idxs])
    print(f"{m:<15} {len(idxs):>3} {taus[idxs].mean():>5.0f}"
          f" {dims[idxs].mean():>5.1f}"
          f" {det.mean():>6.3f} {lam.mean():>6.3f} {entr.mean():>6.3f}")

The table above shows module-level averages.  Box plots reveal the
**full distribution** of RQA measures within each module -- do all
neurons in a group behave similarly, or is there high variability?

In [ ]:
MODULE_SHORT = {
    "event_0": "E1", "event_1": "E2", "event_2": "E3",
    "event_0|1": "E1|E2", "event_0|2": "E1|E3", "event_1|2": "E2|E3",
}

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, measure in zip(axes, ["DET", "LAM", "ENTR"]):
    data_by_mod = []
    labels = []
    for m in module_names:
        vals = [rqa_results[i][measure] for i in groups[m]]
        data_by_mod.append(vals)
        labels.append(MODULE_SHORT[m])

    bp = ax.boxplot(data_by_mod, labels=labels, patch_artist=True)
    for patch, m in zip(bp["boxes"], module_names):
        patch.set_facecolor("#aabbdd")
        patch.set_alpha(0.7)

    ax.set_title(measure, fontsize=12)
    ax.set_xlabel("Module")
    ax.grid(axis="y", alpha=0.3)

axes[0].set_ylabel("Value")
fig.suptitle("RQA measure distributions by module", fontsize=13, y=1.01)
fig.tight_layout()
plt.show()

### 2.3 Pairwise Jaccard similarity

If two neurons respond to the same events, they recur at the same
time points.  The **Jaccard index** quantifies this overlap:

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|}$$

where A and B are the sets of recurrence entries (nonzero elements of
the binary RP).  High Jaccard = neurons share dynamical structure.

We use [`pairwise_jaccard_sparse`](https://driada.readthedocs.io/en/latest/api/recurrence/population.html#driada.recurrence.population.pairwise_jaccard_sparse) to compute all pairs
efficiently via sparse matrix multiplication, then compare
within-module vs between-module similarity.

In [ ]:
# Pairwise Jaccard -- accepts RecurrenceGraph objects directly;
# trim_to_min=True handles different embedded lengths from per-neuron tau/dim
print("Computing pairwise Jaccard similarity...")
sim_matrix = pairwise_jaccard_sparse(per_neuron_rgs, trim_to_min=True)
print(f"{n_neurons} graphs compared")

# Within-module vs between-module
within_vals = []
between_vals = []
for i in range(n_neurons):
    for j in range(i + 1, n_neurons):
        if neuron_modules[i] == neuron_modules[j]:
            within_vals.append(sim_matrix[i, j])
        else:
            between_vals.append(sim_matrix[i, j])

within_vals = np.array(within_vals)
between_vals = np.array(between_vals)
ratio = within_vals.mean() / between_vals.mean() if between_vals.mean() > 0 else float("inf")

print(f"Within-module Jaccard:  {within_vals.mean():.5f}")
print(f"Between-module Jaccard: {between_vals.mean():.5f}")
print(f"Within/between ratio:   {ratio:.2f}x")

### 2.4 Permutation test

Is the within-module similarity genuinely higher, or could it arise
by chance?  A **permutation test** answers this: we shuffle module
labels 1000 times, recompute the within/between ratio each time, and
build a null distribution.  The p-value is the fraction of shuffled
ratios that equal or exceed the observed value.

In [ ]:
pairs_i, pairs_j = np.triu_indices(n_neurons, k=1)
pair_sims = sim_matrix[pairs_i, pairs_j]
all_labels_arr = np.array([neuron_modules[i] for i in range(n_neurons)])

observed_ratio = within_vals.mean() / between_vals.mean()
n_perms = 1000
perm_ratios = np.empty(n_perms)
rng_perm = np.random.default_rng(42)

for p in range(n_perms):
    shuffled = rng_perm.permutation(all_labels_arr)
    same_mask = shuffled[pairs_i] == shuffled[pairs_j]
    w_mean = pair_sims[same_mask].mean()
    b_mean = pair_sims[~same_mask].mean()
    perm_ratios[p] = w_mean / b_mean if b_mean > 0 else 1.0

perm_p_value = float((perm_ratios >= observed_ratio).mean())

print(f"Observed within/between ratio: {observed_ratio:.3f}")
print(f"Null distribution: {perm_ratios.mean():.3f} +/- {perm_ratios.std():.3f}")
print(f"p-value: {perm_p_value:.4f} (1000 permutations)")

# Histogram of null distribution vs observed
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_ratios, bins=40, color="#aabbdd", edgecolor="white",
        linewidth=0.5, label="Null distribution")
ax.axvline(observed_ratio, color="#d62728", linewidth=2, linestyle="--",
           label=f"Observed = {observed_ratio:.3f}")
ax.set_xlabel("Within/between Jaccard ratio")
ax.set_ylabel("Count")
ax.set_title(f"Permutation test (p = {perm_p_value:.4f}, {n_perms} permutations)",
             fontsize=12)
ax.legend(fontsize=9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

### 2.5 Community detection

Finally, we threshold the Jaccard matrix into a network and run
community detection to recover the original modules.

- **Threshold** the Jaccard matrix at the 90th percentile to keep
  only the strongest edges
- Extract the **giant connected component** using [`Network`](https://driada.readthedocs.io/en/latest/api/network/core.html#driada.network.net_base.Network)
- Run **Louvain** community detection -- a greedy algorithm that
  groups nodes to maximize
  [modularity](https://en.wikipedia.org/wiki/Modularity_(networks))
  (the fraction of edges within communities compared to a random
  network)
- Compare to ground truth with the **Adjusted Rand Index (ARI)** --
  a measure of agreement between two clusterings, corrected for
  chance.  ARI = 1.0 is perfect match; ARI near 0 is no better than
  random.

In [ ]:
# Threshold the Jaccard matrix
upper = sim_matrix[np.triu_indices(n_neurons, k=1)]
thr = np.percentile(upper, 90)
sim_thresholded = sim_matrix.copy()
sim_thresholded[sim_thresholded < thr] = 0
np.fill_diagonal(sim_thresholded, 0)
print(f"Threshold: {thr:.5f} (90th percentile)")

# Build network and extract giant CC
net_adj = sp.csr_matrix(sim_thresholded)
net = Network(adj=net_adj, preprocessing="giant_cc", create_nx_graph=True)
print(f"Network: {net.n} nodes, {net.graph.number_of_edges()} edges")

# Louvain community detection
communities = nx_comm.louvain_communities(net.graph, weight="weight", seed=42)
communities = sorted(communities, key=len, reverse=True)
communities = [set(c) for c in communities]
n_detected = len(communities)

# Real modularity
mod_real = nx_comm.modularity(net.graph, communities, weight="weight")

# Shuffled modularity baseline
net_rand = net.randomize(rmode="adj_iom")
comms_rand = nx_comm.louvain_communities(net_rand.graph, weight="weight", seed=42)
mod_rand = nx_comm.modularity(net_rand.graph, comms_rand, weight="weight")

# ARI
nodes_in_net = set()
for comm in communities:
    nodes_in_net.update(comm)

true_labels = []
detected_labels = []
for ci, comm in enumerate(communities):
    for node in comm:
        detected_labels.append(ci)
        true_labels.append(neuron_modules.get(node, "unknown"))

mod_to_int = {m: i for i, m in enumerate(module_names)}
true_int = [mod_to_int.get(t, -1) for t in true_labels]
ari = adjusted_rand_score(true_int, detected_labels)

print(f"{n_detected} communities detected")
print(f"ARI = {ari:.3f}")
print(f"Modularity: {mod_real:.3f} (shuffled: {mod_rand:.3f})")

# Print composition per community
for ci, comm in enumerate(communities):
    mod_counts = {}
    for node in comm:
        m = neuron_modules.get(node, "unknown")
        mod_counts[m] = mod_counts.get(m, 0) + 1
    composition = ", ".join(f"{m}:{c}" for m, c in sorted(mod_counts.items()))
    print(f"  C{ci}: {len(comm):>3} neurons [{composition}]")

### 2.6 Visualizations

Three views of the results: the similarity network colored by
ground-truth module, Jaccard matrices ordered by module vs by
detected community, and a confusion matrix showing how well each
community maps to a ground-truth module.

In [ ]:
MODULE_COLORS = {
    "event_0": "#1a5acd", "event_1": "#ffaa00", "event_2": "#33cc33",
    "event_0|1": "#cc44cc", "event_0|2": "#00dddd", "event_1|2": "#ff4444",
}

legend_order = ["event_0", "event_1", "event_2",
                "event_0|1", "event_0|2", "event_1|2"]

# --- Figure 1: Network graph ---
from matplotlib.lines import Line2D

fig_net, ax_net = plt.subplots(figsize=(7, 7))

pos_init = nx.spectral_layout(net.graph)
pos_net = nx.spring_layout(
    net.graph, pos=pos_init, k=0.02 / np.sqrt(len(net.graph)),
    iterations=200, seed=42)

nx.draw_networkx_edges(net.graph, pos_net, alpha=0.1, ax=ax_net)

node_list_net = list(net.graph.nodes())
node_colors_net = [MODULE_COLORS[neuron_modules[n]] for n in node_list_net]
nx.draw_networkx_nodes(net.graph, pos_net, nodelist=node_list_net,
                        node_color=node_colors_net, node_size=40,
                        edgecolors="k", linewidths=0.3, ax=ax_net, alpha=0.85)

handles = [Line2D([0], [0], marker="o", color="none",
                   markerfacecolor=MODULE_COLORS[g],
                   markeredgecolor="k", markeredgewidth=0.3,
                   markersize=7, label=MODULE_SHORT[g])
           for g in legend_order]
ax_net.legend(handles=handles, fontsize=9, frameon=False,
              loc="lower center", bbox_to_anchor=(0.5, -0.06), ncol=3)
ax_net.set_title(
    f"Jaccard similarity network ({net.n} neurons, "
    f"{net.graph.number_of_edges()} edges, ARI={ari:.2f})",
    fontsize=12)
ax_net.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# --- Figure 2: Jaccard matrices (1x2 subplots) ---
fig_sum, axes_sum = plt.subplots(1, 2, figsize=(14, 6))

upper_tri = sim_matrix[np.triu_indices(n_neurons, k=1)]
vmin_j = np.percentile(upper_tri, 5)
vmax_j = np.percentile(upper_tri, 99)

# Left: ordered by module
mod_order = []
for m in module_names:
    mod_order.extend(sorted(groups[m]))
mod_arr = np.array(mod_order)
sim_by_mod = sim_matrix[np.ix_(mod_arr, mod_arr)]
np.fill_diagonal(sim_by_mod, 0)

im1 = axes_sum[0].imshow(sim_by_mod, cmap="inferno", aspect="auto",
                          vmin=vmin_j, vmax=vmax_j)
cumsum_mod = np.cumsum([0] + [len(groups[m]) for m in module_names])
for b in cumsum_mod[1:-1]:
    axes_sum[0].axhline(b - 0.5, color="white", linewidth=0.8, alpha=0.7)
    axes_sum[0].axvline(b - 0.5, color="white", linewidth=0.8, alpha=0.7)
axes_sum[0].set_xticks([])
axes_sum[0].set_yticks([])
axes_sum[0].set_title("Jaccard similarity (ordered by module)")
plt.colorbar(im1, ax=axes_sum[0], label="Jaccard", shrink=0.8)

# Right: ordered by community
comm_order = []
for comm in communities:
    comm_order.extend(sorted(comm))
missing = [i for i in range(n_neurons) if i not in nodes_in_net]
comm_order.extend(missing)
comm_arr = np.array(comm_order)
sim_by_comm = sim_matrix[np.ix_(comm_arr, comm_arr)]
np.fill_diagonal(sim_by_comm, 0)

im2 = axes_sum[1].imshow(sim_by_comm, cmap="inferno", aspect="auto",
                          vmin=vmin_j, vmax=vmax_j)
cumsum_comm = np.cumsum(
    [0] + [len(c) for c in communities] + [len(missing)])
for b in cumsum_comm[1:-1]:
    axes_sum[1].axhline(b - 0.5, color="white", linewidth=0.8, alpha=0.7)
    axes_sum[1].axvline(b - 0.5, color="white", linewidth=0.8, alpha=0.7)
axes_sum[1].set_xticks([])
axes_sum[1].set_yticks([])
axes_sum[1].set_title("Jaccard similarity (ordered by community)")
plt.colorbar(im2, ax=axes_sum[1], label="Jaccard", shrink=0.8)

fig_sum.tight_layout()
plt.show()

In [ ]:
# --- Figure 3: Confusion matrix ---
n_comm = len(communities)
confusion = np.zeros((len(module_names), n_comm), dtype=int)
for ci, comm in enumerate(communities):
    for node in comm:
        m = neuron_modules.get(node, "unknown")
        if m in module_names:
            mi = module_names.index(m)
            confusion[mi, ci] += 1

fig_conf, ax_conf = plt.subplots(
    figsize=(max(6, n_comm * 0.8 + 2), max(5, len(module_names) * 0.7 + 1)))
im_conf = ax_conf.imshow(confusion, cmap="Blues", aspect="auto")

for i in range(len(module_names)):
    for j in range(n_comm):
        val = confusion[i, j]
        if val > 0:
            color = "white" if val > confusion.max() / 2 else "black"
            ax_conf.text(j, i, str(val), ha="center", va="center",
                         fontsize=10, color=color, fontweight="bold")

ax_conf.set_xticks(range(n_comm))
ax_conf.set_xticklabels([f"C{i}" for i in range(n_comm)], fontsize=9)
ax_conf.set_yticks(range(len(module_names)))
ax_conf.set_yticklabels([MODULE_SHORT.get(m, m) for m in module_names],
                         fontsize=9)
ax_conf.set_xlabel("Detected community", fontsize=11)
ax_conf.set_ylabel("Ground-truth module", fontsize=11)
ax_conf.set_title(f"Community detection confusion matrix (ARI={ari:.3f})",
                   fontsize=12)
plt.colorbar(im_conf, ax=ax_conf, label="Neuron count", shrink=0.8)
fig_conf.tight_layout()
plt.show()

## Further reading

Standalone examples (run directly, no external data needed):

- [recurrence_basic](https://github.com/iabs-neuro/driada/tree/main/examples/recurrence_basic) -- Recurrence fundamentals on classic signals
- [recurrence_population](https://github.com/iabs-neuro/driada/tree/main/examples/recurrence_population) -- Recovering functional modules from population dynamics

API reference:

- [Recurrence module](https://driada.readthedocs.io/en/latest/api/recurrence/index.html) -- [`RecurrenceGraph`](https://driada.readthedocs.io/en/latest/api/recurrence/recurrence_graph.html#driada.recurrence.recurrence_graph.RecurrenceGraph), [`estimate_tau`](https://driada.readthedocs.io/en/latest/api/recurrence/embedding.html#driada.recurrence.embedding.estimate_tau), [`estimate_embedding_dim`](https://driada.readthedocs.io/en/latest/api/recurrence/embedding.html#driada.recurrence.embedding.estimate_embedding_dim), [`compute_rqa`](https://driada.readthedocs.io/en/latest/api/recurrence/rqa.html#driada.recurrence.rqa.compute_rqa)
- [Network module](https://driada.readthedocs.io/en/latest/api/network/index.html) -- [`Network`](https://driada.readthedocs.io/en/latest/api/network/core.html#driada.network.net_base.Network) for graph analysis

Reference: Marwan, N., Romano, M. C., Thiel, M. & Kurths, J. (2007).
Recurrence plots for the analysis of complex systems. *Physics Reports*,
438(5--6), 237--329.

[All examples](https://github.com/iabs-neuro/driada/tree/main/examples)